# 資料清理
移除全空白列及完全重複列，其餘品質問題保留原值並標記。POI 空白不代表行程錯誤，不因此刪除。
真實競賽資料及其結果未包含於公開版本。從 notebooks/ 執行；輸入固定為 ../data/raw.xlsx，不搜尋其他路徑。

`TravelDistance` 以公尺、`TravelTime` 以分鐘計算；時間欄位使用 UTC+8。Excel 時間欄位須已能被 pandas 讀為 datetime。

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

data_path = Path('../data/raw.xlsx')
output_path = Path('../data/yoxi_cleaned.csv')
if not data_path.is_file():
    raise FileNotFoundError('缺少 data/raw.xlsx；私人資料未提供，請先準備已授權的輸入檔。')
df = pd.read_excel(data_path, dtype={'Id': 'string', 'RiderId': 'string'})
required = ['Id', 'RiderId', 'OrderFormCreatedTime_UTC8', 'OrderStartDateTime_UTC8',
            'OrderEndDateTime_UTC8', 'PickUpLatitude', 'PickUpLongitude',
            'DropOffLatitude', 'DropOffLongitude', 'TravelTime', 'TravelDistance', 'PaymentTotalRange']
missing = sorted(set(required) - set(df.columns))
if missing:
    raise ValueError(f'缺少必要欄位：{missing}')
for column in ['OrderFormCreatedTime_UTC8', 'OrderStartDateTime_UTC8', 'OrderEndDateTime_UTC8']:
    if not pd.api.types.is_datetime64_any_dtype(df[column]):
        raise TypeError(f'{column} 必須是 datetime；請核對輸入格式，勿直接猜測時間單位。')

all_empty = df.isna().all(axis=1)
nonempty = df.loc[~all_empty]
clean = nonempty.drop_duplicates().copy()
display(pd.Series({'全空白列': int(all_empty.sum()),
                   '非空白列中的完全重複列': int(nonempty.duplicated().sum()),
                   '清理後列數': len(clean),
                   '保留的重複 Id': int(clean['Id'].duplicated().sum())}))


## 品質檢查
先排除全空白列，再計算完全重複列，避免重複計算刪除數。重複 Id 只檢查，不以 Id 自動去重。
以下檢查與旗標不刪除行程；01 的距離及重疊行程分析另有各自的篩選。

In [ ]:
checks = {'訂單建立時間晚於行程開始': clean['OrderFormCreatedTime_UTC8'] > clean['OrderStartDateTime_UTC8'],
    '行程結束時間早於開始': clean['OrderEndDateTime_UTC8'] < clean['OrderStartDateTime_UTC8'],
    'TravelTime 小於或等於 0': clean['TravelTime'] <= 0, 'TravelDistance 小於或等於 0': clean['TravelDistance'] <= 0,
    '緯度超出 -90 至 90': ~clean['PickUpLatitude'].between(-90, 90) | ~clean['DropOffLatitude'].between(-90, 90),
    '經度超出 -180 至 180': ~clean['PickUpLongitude'].between(-180,
    180) | ~clean['DropOffLongitude'].between(-180, 180)}

display(pd.Series({name: int(mask.sum()) for name, mask in checks.items()}, name="待檢查筆數"))


In [ ]:
lat1 = np.radians(clean['PickUpLatitude'])
lon1 = np.radians(clean['PickUpLongitude'])
lat2 = np.radians(clean['DropOffLatitude'])
lon2 = np.radians(clean['DropOffLongitude'])
a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
straight_line_m = 2 * 6371000 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))
recorded_speed_kmh = clean['TravelDistance'] / 1000 / (clean['TravelTime'] / 60)

elapsed_minutes = (clean['OrderEndDateTime_UTC8'] - clean['OrderStartDateTime_UTC8']).dt.total_seconds() / 60
time_difference = (clean['TravelTime'] - elapsed_minutes).abs()
display(time_difference.describe(percentiles=[0.5, 0.95, 0.99, 0.999]))


直線距離使用 Haversine，地球半徑 6,371,000 公尺。平均速度由記錄距離與記錄分鐘數計算。
旗標門檻：零距離、時間差超過 2 分鐘、時間超過 180 分鐘、距離超過 200 公里、速度超過 150 km/h；直線距離超過 1 公里且記錄距離低於其 80% 也標記。
`TravelTime` 與起訖時間的業務定義可能不同，保留原值，不依取整方式重算。
座標另檢查合法經緯度範圍及北緯 21–27、東經 118–123 的寬鬆框；框內不代表道路或位置正確。

In [ ]:
clean['qa_zero_distance'] = clean['TravelDistance'].eq(0)
clean['qa_time_gap_over_2min'] = time_difference.gt(2)
clean['qa_trip_over_180min'] = clean['TravelTime'].gt(180)
clean['qa_distance_over_200km'] = clean['TravelDistance'].gt(200000)
clean['qa_speed_over_150kmh'] = recorded_speed_kmh.gt(150)
clean['qa_distance_below_straight_line'] = (straight_line_m > 1000) & (clean['TravelDistance'] < straight_line_m * 0.8)
clean['qa_missing_payment_range'] = clean['PaymentTotalRange'].isna()
qa_columns = [col for col in clean.columns if col.startswith('qa_')]
qa_summary = pd.DataFrame({'筆數': clean[qa_columns].sum(),
    '占全部比例(%)': (clean[qa_columns].mean() * 100).round(3)})
display(qa_summary)

pickup_outside = ~(clean['PickUpLatitude'].between(21, 27) & clean['PickUpLongitude'].between(118, 123))
dropoff_outside = ~(clean['DropOffLatitude'].between(21, 27) & clean['DropOffLongitude'].between(118, 123))
display(pd.Series({'上車點超出檢查框': int(pickup_outside.sum()),
                   '下車點超出檢查框': int(dropoff_outside.sum()),
                   '任一端超出檢查框': int((pickup_outside | dropoff_outside).sum())}))
clean.to_csv(output_path, index=False, encoding='utf-8-sig')
print(f'清理檔已寫入：{output_path}')
